In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import time

# ============================================================
# SEARCH INDUSTRY + MAKE CHART
# Memory optimized: chunk read, aggregate only, no saving
# ============================================================

FILE = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile" / "business_startup_ALL_IN_ONE.csv"

SEARCH_KEYWORD = "mining"   # change this
STATE_FILTER = "U.S. totals"       # use "U.S. totals" for national chart
CHUNKSIZE = 300_000

industry_label_map = {
    "NAICS11": "Agriculture, Forestry, Fishing and Hunting",
    "NAICS21": "Mining, Quarrying, and Oil/Gas Extraction",
    "NAICS22": "Utilities",
    "NAICS23": "Construction",
    "NAICS42": "Wholesale Trade",
    "NAICS51": "Information",
    "NAICS52": "Finance and Insurance",
    "NAICS53": "Real Estate and Rental and Leasing",
    "NAICS54": "Professional, Scientific, and Technical Services",
    "NAICS55": "Management of Companies and Enterprises",
    "NAICS56": "Administrative and Support and Waste Management",
    "NAICS61": "Educational Services",
    "NAICS62": "Health Care and Social Assistance",
    "NAICS71": "Arts, Entertainment, and Recreation",
    "NAICS72": "Accommodation and Food Services",
    "NAICS81": "Other Services except Public Administration",
    "NAICSMNF": "Manufacturing",
    "NAICSRET": "Retail Trade",
    "NAICSTW": "Transportation and Warehousing",
    "NONAICS": "Not classified",
    "TOTAL": "Total / all industries",
}

usecols = [
    "source",
    "dataset",
    "year",
    "state_name",
    "industry_code",
    "industry_name",
    "metric_name",
    "value",
]

keep_metrics = [
    "Establishment Births",
    "Business Births",
    "Firm Births",
    "Openings",
    "Closings",
    "Gross Job Gains",
    "Gross Job Losses",
    "Job Creation",
    "Job Destruction",
]

print("Checking file:")
print(FILE)

if not FILE.exists():
    raise FileNotFoundError(f"File not found:\n{FILE}")

print("File exists.")
print("Searching:", SEARCH_KEYWORD)
print("State filter:", STATE_FILTER)
print()

start = time.time()
rows_checked = 0
matched_rows = 0

agg_parts = []
unique_industries = {}

for i, chunk in enumerate(pd.read_csv(FILE, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False), start=1):
    rows_checked += len(chunk)

    chunk["industry_code"] = chunk["industry_code"].astype(str).str.strip()
    chunk["industry_name"] = chunk["industry_name"].astype(str).str.strip()
    chunk["state_name"] = chunk["state_name"].astype(str).str.strip()
    chunk["metric_name"] = chunk["metric_name"].astype(str).str.strip()
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    # Fix NAICS labels
    chunk["industry_name_clean"] = chunk["industry_code"].map(industry_label_map).fillna(chunk["industry_name"])

    # Search industry code + industry name
    search_text = (
        chunk["industry_code"].str.lower()
        + " "
        + chunk["industry_name_clean"].str.lower()
    )

    mask = search_text.str.contains(SEARCH_KEYWORD.lower(), na=False)

    # National chart only
    mask = mask & (chunk["state_name"] == STATE_FILTER)

    # Keep useful business metrics only
    mask = mask & chunk["metric_name"].isin(keep_metrics)

    found = chunk.loc[mask].copy()
    matched_rows += len(found)

    if not found.empty:
        # collect unique matching industries
        temp_unique = found[["industry_code", "industry_name_clean"]].drop_duplicates()
        for _, row in temp_unique.iterrows():
            unique_industries[row["industry_code"]] = row["industry_name_clean"]

        # aggregate inside chunk
        temp_agg = (
            found.groupby(["year", "metric_name"], as_index=False)["value"]
            .sum()
        )
        agg_parts.append(temp_agg)

    if i % 5 == 0:
        elapsed = round(time.time() - start, 1)
        print(f"Chunk {i} | rows checked {rows_checked:,} | matched rows {matched_rows:,} | elapsed {elapsed}s")

print()
print("DONE")
print("Rows checked:", f"{rows_checked:,}")
print("Matched rows:", f"{matched_rows:,}")
print()

# ============================================================
# SHOW UNIQUE INDUSTRIES FOUND
# ============================================================

unique_df = pd.DataFrame(
    sorted(unique_industries.items()),
    columns=["industry_code", "industry_name_clean"]
)

print("Unique matching industries:")
display(unique_df)

# ============================================================
# FINAL AGGREGATED TABLE + CHART
# ============================================================

if not agg_parts:
    print("No matching data found.")
else:
    agg = pd.concat(agg_parts, ignore_index=True)

    final = (
        agg.groupby(["year", "metric_name"], as_index=False)["value"]
        .sum()
        .sort_values(["year", "metric_name"])
    )

    print("Aggregated chart data:")
    display(final)

    chart_df = final.pivot(index="year", columns="metric_name", values="value").fillna(0)

    # Make chart
    plt.figure(figsize=(14, 7))

    for col in chart_df.columns:
        plt.plot(chart_df.index, chart_df[col], marker="o", label=col)

    plt.title(f"{SEARCH_KEYWORD.title()} business/job flow metrics over time")
    plt.xlabel("Year")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path
import pandas as pd
import time

# ============================================================
# PRINT SEARCH OPTIONS FROM business_startup_ALL_IN_ONE.csv
# Memory optimized: chunk read, no saving
# ============================================================

FILE = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile" / "business_startup_ALL_IN_ONE.csv"

CHUNKSIZE = 300_000

industry_label_map = {
    "NAICS11": "Agriculture, Forestry, Fishing and Hunting",
    "NAICS21": "Mining, Quarrying, and Oil/Gas Extraction",
    "NAICS22": "Utilities",
    "NAICS23": "Construction",
    "NAICS42": "Wholesale Trade",
    "NAICS51": "Information",
    "NAICS52": "Finance and Insurance",
    "NAICS53": "Real Estate and Rental and Leasing",
    "NAICS54": "Professional, Scientific, and Technical Services",
    "NAICS55": "Management of Companies and Enterprises",
    "NAICS56": "Administrative and Support and Waste Management",
    "NAICS61": "Educational Services",
    "NAICS62": "Health Care and Social Assistance",
    "NAICS71": "Arts, Entertainment, and Recreation",
    "NAICS72": "Accommodation and Food Services",
    "NAICS81": "Other Services except Public Administration",
    "NAICSMNF": "Manufacturing",
    "NAICSRET": "Retail Trade",
    "NAICSTW": "Transportation and Warehousing",
    "NONAICS": "Not classified",
    "TOTAL": "Total / all industries",
}

usecols = [
    "source",
    "dataset",
    "year",
    "state_name",
    "industry_code",
    "industry_name",
    "metric_name",
    "geo_level",
]

print("Checking file:")
print(FILE)

if not FILE.exists():
    raise FileNotFoundError(f"File not found:\n{FILE}")

print("File exists.")
print()

start = time.time()
rows_checked = 0

industries = {}
metrics = set()
states = set()
datasets = set()
sources = set()
geo_levels = set()
years = set()

for i, chunk in enumerate(pd.read_csv(FILE, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False), start=1):
    rows_checked += len(chunk)

    for col in usecols:
        chunk[col] = chunk[col].astype(str).str.strip()

    chunk["industry_name_clean"] = chunk["industry_code"].map(industry_label_map).fillna(chunk["industry_name"])

    temp_industries = chunk[["industry_code", "industry_name_clean"]].drop_duplicates()

    for _, row in temp_industries.iterrows():
        industries[row["industry_code"]] = row["industry_name_clean"]

    metrics.update(chunk["metric_name"].dropna().unique())
    states.update(chunk["state_name"].dropna().unique())
    datasets.update(chunk["dataset"].dropna().unique())
    sources.update(chunk["source"].dropna().unique())
    geo_levels.update(chunk["geo_level"].dropna().unique())
    years.update(chunk["year"].dropna().unique())

    if i % 5 == 0:
        elapsed = round(time.time() - start, 1)
        print(f"Chunk {i} | rows checked {rows_checked:,} | elapsed {elapsed}s")

print()
print("DONE")
print("Rows checked:", f"{rows_checked:,}")
print()

# ============================================================
# DISPLAY OPTIONS
# ============================================================

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

industry_df = pd.DataFrame(
    sorted(industries.items()),
    columns=["industry_code", "industry_name_clean"]
)

metric_df = pd.DataFrame(
    sorted([x for x in metrics if x and x.lower() != "nan"]),
    columns=["metric_name"]
)

state_df = pd.DataFrame(
    sorted([x for x in states if x and x.lower() != "nan"]),
    columns=["state_name"]
)

dataset_df = pd.DataFrame(
    sorted([x for x in datasets if x and x.lower() != "nan"]),
    columns=["dataset"]
)

source_df = pd.DataFrame(
    sorted([x for x in sources if x and x.lower() != "nan"]),
    columns=["source"]
)

geo_df = pd.DataFrame(
    sorted([x for x in geo_levels if x and x.lower() != "nan"]),
    columns=["geo_level"]
)

year_df = pd.DataFrame(
    sorted([x for x in years if x and x.lower() != "nan"]),
    columns=["year"]
)

print("SEARCHABLE INDUSTRIES")
display(industry_df)

print("SEARCHABLE METRICS")
display(metric_df)

print("SEARCHABLE STATES")
display(state_df)

print("SEARCHABLE DATASETS")
display(dataset_df)

print("SEARCHABLE SOURCES")
display(source_df)

print("SEARCHABLE GEO LEVELS")
display(geo_df)

print("YEARS AVAILABLE")
display(year_df)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import time

# ============================================================
# SEARCH INDUSTRY + MAKE CHART
# Memory optimized: chunk read, aggregate only, no saving
# Better search: multiple terms + no regex weirdness
# ============================================================

FILE = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile" / "business_startup_ALL_IN_ONE.csv"

# Better than one long exact search
SEARCH_TERMS = ["mining", "NAICS21", "100010", "natural resources"]

# For national BLS rows, use "U.S. totals"
# If you get no matches, change this to None to see all states/datasets
STATE_FILTER = "U.S. totals"

CHUNKSIZE = 300_000

industry_label_map = {
    "NAICS11": "Agriculture, Forestry, Fishing and Hunting",
    "NAICS21": "Mining, Quarrying, and Oil/Gas Extraction",
    "NAICS22": "Utilities",
    "NAICS23": "Construction",
    "NAICS42": "Wholesale Trade",
    "NAICS51": "Information",
    "NAICS52": "Finance and Insurance",
    "NAICS53": "Real Estate and Rental and Leasing",
    "NAICS54": "Professional, Scientific, and Technical Services",
    "NAICS55": "Management of Companies and Enterprises",
    "NAICS56": "Administrative and Support and Waste Management",
    "NAICS61": "Educational Services",
    "NAICS62": "Health Care and Social Assistance",
    "NAICS71": "Arts, Entertainment, and Recreation",
    "NAICS72": "Accommodation and Food Services",
    "NAICS81": "Other Services except Public Administration",
    "NAICSMNF": "Manufacturing",
    "NAICSRET": "Retail Trade",
    "NAICSTW": "Transportation and Warehousing",
    "NONAICS": "Not classified",
    "TOTAL": "Total / all industries",
}

usecols = [
    "source",
    "dataset",
    "year",
    "state_name",
    "industry_code",
    "industry_name",
    "metric_name",
    "value",
]

keep_metrics = [
    "Establishment Births",
    "Business Births",
    "Firm Births",
    "Openings",
    "Closings",
    "Gross Job Gains",
    "Gross Job Losses",
    "Job Creation",
    "Job Destruction",
]

print("Checking file:")
print(FILE)

if not FILE.exists():
    raise FileNotFoundError(f"File not found:\n{FILE}")

print("File exists.")
print("Search terms:", SEARCH_TERMS)
print("State filter:", STATE_FILTER)
print()

start = time.time()
rows_checked = 0
matched_before_state = 0
matched_after_state = 0
matched_after_metric = 0

agg_parts = []
unique_industries = {}
unique_states_found = set()
unique_metrics_found = set()

for i, chunk in enumerate(pd.read_csv(FILE, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False), start=1):
    rows_checked += len(chunk)

    chunk["industry_code"] = chunk["industry_code"].fillna("").astype(str).str.strip()
    chunk["industry_name"] = chunk["industry_name"].fillna("").astype(str).str.strip()
    chunk["state_name"] = chunk["state_name"].fillna("").astype(str).str.strip()
    chunk["metric_name"] = chunk["metric_name"].fillna("").astype(str).str.strip()
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    # Fix NAICS labels
    chunk["industry_name_clean"] = chunk["industry_code"].map(industry_label_map).fillna(chunk["industry_name"])

    # Search code + clean name
    search_text = (
        chunk["industry_code"].str.lower()
        + " "
        + chunk["industry_name_clean"].str.lower()
    )

    search_mask = pd.Series(False, index=chunk.index)

    for term in SEARCH_TERMS:
        search_mask = search_mask | search_text.str.contains(term.lower(), regex=False, na=False)

    matched_before_state += search_mask.sum()

    # Keep states/metrics found before filters so you can debug
    if search_mask.any():
        unique_states_found.update(chunk.loc[search_mask, "state_name"].dropna().unique())
        unique_metrics_found.update(chunk.loc[search_mask, "metric_name"].dropna().unique())

        temp_unique = chunk.loc[search_mask, ["industry_code", "industry_name_clean"]].drop_duplicates()
        for _, row in temp_unique.iterrows():
            unique_industries[row["industry_code"]] = row["industry_name_clean"]

    # State filter
    if STATE_FILTER is not None:
        state_mask = chunk["state_name"] == STATE_FILTER
    else:
        state_mask = pd.Series(True, index=chunk.index)

    after_state_mask = search_mask & state_mask
    matched_after_state += after_state_mask.sum()

    # Metric filter
    metric_mask = chunk["metric_name"].isin(keep_metrics)

    final_mask = after_state_mask & metric_mask
    found = chunk.loc[final_mask].copy()
    matched_after_metric += len(found)

    if not found.empty:
        temp_agg = (
            found.groupby(["year", "metric_name"], as_index=False)["value"]
            .sum()
        )
        agg_parts.append(temp_agg)

    if i % 5 == 0:
        elapsed = round(time.time() - start, 1)
        print(
            f"Chunk {i} | rows checked {rows_checked:,} | "
            f"matched before state {matched_before_state:,} | "
            f"after state {matched_after_state:,} | "
            f"after metric {matched_after_metric:,} | elapsed {elapsed}s"
        )

print()
print("DONE")
print("Rows checked:", f"{rows_checked:,}")
print("Matched before state filter:", f"{matched_before_state:,}")
print("Matched after state filter:", f"{matched_after_state:,}")
print("Matched after metric filter:", f"{matched_after_metric:,}")
print()

# ============================================================
# SHOW UNIQUE INDUSTRIES FOUND
# ============================================================

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

unique_df = pd.DataFrame(
    sorted(unique_industries.items()),
    columns=["industry_code", "industry_name_clean"]
)

print("Unique matching industries:")
display(unique_df)

print("States found before state filter:")
display(pd.DataFrame(sorted([x for x in unique_states_found if x]), columns=["state_name"]))

print("Metrics found before metric filter:")
display(pd.DataFrame(sorted([x for x in unique_metrics_found if x]), columns=["metric_name"]))

# ============================================================
# FINAL AGGREGATED TABLE + CHART
# ============================================================

if not agg_parts:
    print("No matching chart data found.")
    print()
    print("Try this:")
    print('STATE_FILTER = None')
    print()
    print("Or try:")
    print('SEARCH_TERMS = ["mining"]')
else:
    agg = pd.concat(agg_parts, ignore_index=True)

    final = (
        agg.groupby(["year", "metric_name"], as_index=False)["value"]
        .sum()
        .sort_values(["year", "metric_name"])
    )

    print("Aggregated chart data:")
    display(final)

    chart_df = final.pivot(index="year", columns="metric_name", values="value").fillna(0)

    plt.figure(figsize=(14, 7))

    for col in chart_df.columns:
        plt.plot(chart_df.index, chart_df[col], marker="o", label=col)

    plt.title("Mining / natural resources business and job flow metrics over time")
    plt.xlabel("Year")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

WHAT TO SEARCH IN business_startup_ALL_IN_ONE.csv

Your file has 4 main search areas:

1. INDUSTRY
This tells you what type of business/company.

2. METRIC
This tells you what number you are measuring.

3. DATASET / SOURCE
This tells you where the data came from.

4. YEAR
Your file has years 1978 to 2026.


============================================================
INDUSTRIES YOU CAN SEARCH
============================================================

TOTAL / ALL BUSINESS
Search:
total
total private
all industries

Use for:
overall U.S. business trend


GOODS-PRODUCING
Search:
goods-producing

Use for:
manufacturing, construction, mining, agriculture type industries


SERVICE-PROVIDING
Search:
service-providing

Use for:
service economy, tech, finance, health, education, hospitality


AGRICULTURE
Search:
agriculture
forestry
fishing
hunting

Use for:
agriculture, farming, natural resources


MINING / OIL / GAS
Search:
mining
natural resources
oil
gas
quarrying

Use for:
mining, oil/gas, natural resource industries


CONSTRUCTION
Search:
construction

Use for:
civil engineering, construction companies, building industry


MANUFACTURING
Search:
manufacturing

Use for:
mechanical engineering, chemical engineering, industrial engineering, factories, production


WHOLESALE TRADE
Search:
wholesale

Use for:
distribution, supply chain, wholesale business


RETAIL TRADE
Search:
retail

Use for:
stores, retail business, small business, sales companies


TRANSPORTATION / WAREHOUSING
Search:
transportation
warehousing

Use for:
logistics, trucking, shipping, warehouse jobs, supply chain


UTILITIES
Search:
utilities

Use for:
electric, water, power, energy utility companies


INFORMATION / TECH
Search:
information

Use for:
computer science, software, IT, tech, data, media, telecom


FINANCE
Search:
financial
finance
insurance

Use for:
accounting, finance, banking, insurance, economics


REAL ESTATE
Search:
real estate
rental
leasing

Use for:
property, housing, rental companies, real estate business


PROFESSIONAL / BUSINESS SERVICES
Search:
professional
business services

Use for:
office jobs, consulting, engineering services, company services


SCIENTIFIC / TECHNICAL SERVICES
Search:
scientific
technical
professional

Use for:
data science, engineering, research, computer-related professional work


MANAGEMENT OF COMPANIES
Search:
management
companies
enterprises

Use for:
corporate offices, headquarters, company management


ADMINISTRATIVE SUPPORT
Search:
administrative
support
waste management

Use for:
office support, staffing, business support services


EDUCATION
Search:
education
educational

Use for:
schools, colleges, education jobs


HEALTH
Search:
health
health care
social assistance

Use for:
biomedical, healthcare, nursing, medical companies, hospitals


EDUCATION AND HEALTH
Search:
education and health
education
health

Use for:
broad education + healthcare industry together


ARTS / ENTERTAINMENT
Search:
arts
entertainment
recreation

Use for:
creative industry, entertainment, events, recreation


FOOD / HOTEL
Search:
accommodation
food
hotel
restaurant

Use for:
restaurant, hotel, tourism, food service


LEISURE / HOSPITALITY
Search:
leisure
hospitality
accommodation
food

Use for:
tourism, restaurants, hotels, travel industry


OTHER SERVICES
Search:
other services

Use for:
repair, personal services, laundry, salons, miscellaneous services


NOT CLASSIFIED
Search:
not classified

Use for:
records that do not have a clear industry


============================================================
BEST SEARCHES FOR YOUR DEGREE / JOB / COMPANY PROJECT
============================================================

Computer science / software / IT:
Search:
information

Data science / research / engineering services:
Search:
scientific
technical
professional

Mechanical engineering:
Search:
manufacturing

Chemical engineering:
Search:
manufacturing

Industrial engineering:
Search:
manufacturing

Civil engineering:
Search:
construction

Biomedical / healthcare:
Search:
health
health care

Business / accounting / finance:
Search:
finance
financial
insurance

Real estate / rental business:
Search:
real estate
rental
leasing

Restaurant / hotel / tourism:
Search:
food
hotel
restaurant
hospitality

Supply chain / logistics:
Search:
transportation
warehousing

Small business / stores:
Search:
retail


============================================================
METRICS YOU CAN SEARCH
============================================================

OPENINGS
Search:
Openings

Meaning:
business establishments that opened

Good for:
new company activity


CLOSINGS
Search:
Closings

Meaning:
business establishments that closed

Good for:
business failure / decline proxy


ESTABLISHMENT BIRTHS
Search:
Establishment Births

Meaning:
new establishments born

Good for:
new business creation


ESTABLISHMENT DEATHS
Search:
Establishment Deaths

Meaning:
establishments that died

Good for:
business failure / shutdown trend


GROSS JOB GAINS
Search:
Gross Job Gains

Meaning:
jobs added by expanding/opening establishments

Good for:
hiring growth


GROSS JOB LOSSES
Search:
Gross Job Losses

Meaning:
jobs lost by shrinking/closing establishments

Good for:
job decline / layoffs


FIRMS
Search:
firms

Meaning:
number of firms / companies

Good for:
how many companies exist


EMPLOYMENT
Search:
emp

Meaning:
employment count

Good for:
how many workers are in that industry


JOB CREATION
Search:
job_creation

Meaning:
jobs created

Good for:
growth from companies


JOB CREATION BIRTHS
Search:
job_creation_births

Meaning:
jobs created by new firms

Good for:
startup/new company job creation


NET JOB CREATION
Search:
net_job_creation

Meaning:
jobs created minus jobs lost

Good for:
overall job growth or decline


FIRM DEATHS
Search:
firmdeath_firms

Meaning:
number of firms that died

Good for:
company failure proxy


EMPLOYMENT LOST FROM FIRM DEATHS
Search:
firmdeath_emp

Meaning:
jobs lost because firms died

Good for:
impact of company failure


BUSINESS APPLICATIONS
Search:
BA_BA

Meaning:
business applications

Good for:
people applying to start businesses


HIGH-PROPENSITY BUSINESS APPLICATIONS
Search:
BA_HBA

Meaning:
business applications likely to become real businesses

Good for:
stronger startup signal


BUSINESS FORMATIONS
Search:
BF_BF4Q
BF_BF8Q

Meaning:
projected business formations within 4 or 8 quarters

Good for:
new business formation forecast


BANKRUPTCY
Search:
F-2 Total row numeric value

Meaning:
bankruptcy filing data

Good for:
business/legal failure trend, but less clean by industry


============================================================
BEST METRICS TO CHART FIRST
============================================================

For business openings vs closings:
Openings
Closings
Establishment Births
Establishment Deaths

For job growth:
Gross Job Gains
Gross Job Losses

For company count:
firms
emp

For startup/new business applications:
BA_BA
BA_HBA
BA_CBA
BA_WBA

For business formation forecast:
BF_BF4Q
BF_BF8Q

For failure proxy:
firmdeath_firms
firmdeath_emp


============================================================
DATASETS / SOURCES
============================================================

BLS BED/BDM
Good for:
Openings
Closings
Gross Job Gains
Gross Job Losses
Establishment Births
Establishment Deaths

Best use:
job flow and business opening/closing trends


Census BDS
Good for:
firms
emp
job_creation
firmdeath_firms
firmdeath_emp

Best use:
company count, employment, firm births/deaths, business survival/failure proxy


Census BFS
Good for:
BA_BA
BA_HBA
BF_BF4Q
BF_BF8Q

Best use:
business applications and projected new business formations


U.S. Courts F-2
Good for:
bankruptcy numbers

Best use:
bankruptcy trend, but not as directly connected to degree/industry search


============================================================
YEARS AVAILABLE
============================================================

Your file has:
1978 to 2026

But not every dataset has every year.

For long historical company trend:
Use Census BDS

For openings/closings/job gains/job losses:
Use BLS BED/BDM

For newest business applications:
Use Census BFS

For bankruptcy:
Use U.S. Courts F-2


============================================================
SIMPLE SEARCH RECOMMENDATIONS
============================================================

Best first search:
manufacturing

Best for computer science:
information

Best for data science:
scientific
technical
professional

Best for biomedical:
health

Best for civil engineering:
construction

Best for business/accounting:
finance

Best for startup/business trend:
total

Best first metrics:
Openings
Closings
Gross Job Gains
Gross Job Losses
Establishment Births
Establishment Deaths


============================================================
EXAMPLE MEANINGS
============================================================

Search manufacturing + Openings:
How many manufacturing establishments opened over time.

Search manufacturing + Closings:
How many manufacturing establishments closed over time.

Search information + Gross Job Gains:
How many jobs were gained in tech/information industries.

Search health + Establishment Births:
How many new health-related establishments were created.

Search construction + Gross Job Losses:
How many jobs were lost in construction.

Search finance + firms:
How many finance/insurance firms existed.

Search total + BA_BA:
How many total business applications happened.

Search total + BF_BF4Q:
Projected business formations within 4 quarters.

Search manufacturing + firmdeath_firms:
How many manufacturing firms died.

Search manufacturing + firmdeath_emp:
How many jobs were lost because manufacturing firms died.


============================================================
IMPORTANT NOTE
============================================================

Do not search long full names like:
Mining, Quarrying, and Oil/Gas Extraction

Better search:
mining
oil
gas
natural resources

Do not search long full names like:
Professional, Scientific, and Technical Services

Better search:
professional
scientific
technical

Do not search long full names like:
Accommodation and Food Services

Better search:
food
hotel
restaurant
hospitality